# 🛡️ AI Data Poisoning Defense — Training on Google Colab

Цей ноутбук тренує систему захисту від отруєння даних і публікує результати на HuggingFace Hub.

**План:**
1. Перевірити GPU
2. Встановити залежності
3. Завантажити код (ZIP upload або Git clone)
4. Натренувати Detector + Baseline + Protected моделі
5. Опублікувати ваги на HF Models

**⚠️ Перед запуском:** Runtime → Change runtime type → **T4 GPU**


## 1. Перевірка GPU

In [ ]:
# Має показати T4 (або кращий GPU)
!nvidia-smi

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Встановлення залежностей

In [ ]:
!pip install -q datasets>=2.14.0 huggingface_hub>=0.20.0
# torch/torchvision/numpy/pillow вже встановлені у Colab

## 3. Завантаження коду

**Варіант A (рекомендую): ZIP-архів**
- Запакуй локальну папку `poison_defense/` у ZIP
- Запусти комірку нижче і завантаж її через діалог

**Варіант B: Git clone**
- Якщо ти запушив код на GitHub, розкоментуй другу комірку


In [ ]:
# === ВАРІАНТ A: завантаження ZIP ===
from google.colab import files
import zipfile, os, shutil

# Очищаємо попередню папку якщо є
if os.path.exists('poison_defense'):
    shutil.rmtree('poison_defense')

print("📤 Завантаж ZIP з папкою poison_defense:")
uploaded = files.upload()  # відкриє діалог вибору файла

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

# Якщо ZIP розпакувався в підпапку — знаходимо ту, де лежить train.py
if not os.path.exists('poison_defense/train.py'):
    for root, dirs, files_ in os.walk('.'):
        if 'train.py' in files_ and 'poison_generator.py' in files_:
            # знайшли — переміщаємо в /content/poison_defense
            if root != './poison_defense':
                if os.path.exists('poison_defense'):
                    shutil.rmtree('poison_defense')
                shutil.move(root, 'poison_defense')
            break

print("\n📁 Вміст poison_defense/:")
!ls -la poison_defense/

In [ ]:
# === ВАРІАНТ B: Git clone (якщо є GitHub репо) ===
# !git clone https://github.com/YOUR_USERNAME/poison-defense.git poison_defense
# !ls -la poison_defense/

In [ ]:
# Переходимо в робочу папку
%cd poison_defense
!ls

## 4. Тренування

Параметри підібрані для T4 GPU. На CIFAR-10:
- Тренування Detector: ~3 хв
- Тренування Baseline + Protected: ~10 хв
- **Загалом: ~15 хв**

Якщо хочеш швидший прогон для перевірки — зменши `--epochs_*`.


In [ ]:
!python train.py \
    --dataset cifar10 \
    --epochs_detector 5 \
    --epochs_classifier 15 \
    --batch_size 256 \
    --poison_ratio 0.3 \
    --lr 1e-3 \
    --num_workers 2

## 5. Перевірка чекпойнтів

Після успішного тренування з'являться 3 файли в `checkpoints/`:
- `detector.pt` — навчений детектор
- `baseline.pt` — модель без захисту
- `protected.pt` — модель з захистом


In [ ]:
!ls -lh checkpoints/

## 6. Логін до HuggingFace

Тобі потрібен HF token (Write access):
1. Зайди на https://huggingface.co/settings/tokens
2. Натисни **"New token"** → роль **"Write"**
3. Скопіюй і встав у поле, що з'явиться нижче


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 7. Публікація на HuggingFace Hub

Замін `YOUR_USERNAME` нижче на свій логін на HF.


In [ ]:
HF_USERNAME = "YOUR_USERNAME"  # ⚠️ ЗАМІНИ НА СВІЙ
REPO_NAME = "poison-defense-cifar10"

# Запускаємо push-скрипт
!python push_to_hub.py \
    --username {HF_USERNAME} \
    --repo_name {REPO_NAME} \
    --checkpoint_dir ./checkpoints \
    --dataset cifar10

## 8. Готово! 🎉

Твоя модель тепер на HF Hub:
**https://huggingface.co/YOUR_USERNAME/poison-defense-cifar10**

### Наступні кроки:
1. **Створити Space-демо** — використай `app.py` що йде з кодом
2. **Поекспериментувати** — спробуй інші датасети:
   ```
   !python train.py --dataset cifar100 --epochs_classifier 20
   !python train.py --dataset fashion_mnist
   ```
3. **Додати meta-learning** — для адаптації до нових типів атак
